In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans,DBSCAN
from sklearn.feature_extraction.text import TfidfVectorizer
from fancyimpute import IterativeImputer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull
from collections import defaultdict
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
import pickle

import os 
os.environ['OMP_NUMRHREADS']='1'

In [2]:
price_train=pd.read_csv('data/ruc_Class25Q2_train_price.csv')
price_test=pd.read_csv('data/ruc_Class25Q2_test_price.csv')

C:\Users\oasis\AppData\Local\Temp\ipykernel_47516\3997896161.py:1: DtypeWarning: Columns (3,32,34,43,46,49,51) have mixed types. Specify dtype option on import or set low_memory=False.
  price_train=pd.read_csv('data/ruc_Class25Q2_train_price.csv')
C:\Users\oasis\AppData\Local\Temp\ipykernel_47516\3997896161.py:2: DtypeWarning: Columns (4,32) have mixed types. Specify dtype option on import or set low_memory=False.
  price_test=pd.read_csv('data/ruc_Class25Q2_test_price.csv')


In [3]:
# 保存原始索引
price_train_original_index = price_train.index.copy()
price_test_original_index = price_test.index.copy()

#安全提取目标变量和ID
target_Price=price_train['Price']
price_train=price_train.drop('Price',axis=1)
price_train=price_train.drop(['房屋朝向','区县','板块_comm','环线位置','抵押信息'
                              ,'产权描述','物业类别','物业办公电话','coord_x','coord_y'
                             ,'房屋优势','核心卖点','户型介绍','周边配套','交通出行','客户反馈'], axis=1)
price_train.columns = price_train.columns.str.strip()
price_train.columns = [col.replace(' ', '') for col in price_train.columns]

price_test_id = price_test['ID'].copy() 
price_test=price_test.drop('ID',axis=1)
price_test=price_test.drop(['房屋朝向','区县','板块_comm','环线位置','抵押信息'
                            ,'产权描述','物业类别','物业办公电话','coord_x','coord_y'
                           ,'房屋优势','核心卖点','户型介绍','周边配套','交通出行','客户反馈'], axis=1)
price_test.columns = price_test.columns.str.strip()
price_test.columns = [col.replace(' ', '') for col in price_test.columns]

In [4]:
price_train.describe()

,城市,区域,板块,lon,lat,年份,容积率,停车位
count,103871.000000,103871.000000,103871.000000,103871.000000,103871.000000,103871.000000,70717.000000,69303.000000
mean,3.816272,65.564989,604.388405,114.790564,32.181145,2020.913200,2.780097,1158.042264
std,3.356300,33.694763,334.938598,5.837851,5.955405,0.845009,1.607581,1301.382856
min,0.000000,3.000000,1.000000,103.507243,23.025853,2015.000000,0.020000,1.000000
25%,2.000000,35.000000,314.000000,107.620335,30.379362,2021.000000,1.800000,312.000000
50%,3.000000,68.000000,602.000000,115.552921,32.136038,2021.000000,2.500000,734.000000
75%,6.000000,88.000000,909.000000,118.046991,37.699695,2021.000000,3.200000,1515.000000
max,11.000000,131.000000,1186.000000,122.966669,42.189559,2022.000000,30.000000,8700.000000


In [5]:
price_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103871 entries, 0 to 103870
Data columns (total 38 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   城市         103871 non-null  int64  
 1   区域         103871 non-null  float64
 2   板块         103871 non-null  float64
 3   环线         40419 non-null   object 
 4   房屋户型       103291 non-null  object 
 5   所在楼层       103871 non-null  object 
 6   建筑面积       103871 non-null  object 
 7   套内面积       35984 non-null   object 
 8   建筑结构       103291 non-null  object 
 9   装修情况       103291 non-null  object 
 10  梯户比例       101252 non-null  object 
 11  配备电梯       91520 non-null   object 
 12  别墅类型       1443 non-null    object 
 13  交易时间       103871 non-null  object 
 14  交易权属       103871 non-null  object 
 15  上次交易       78422 non-null   object 
 16  房屋用途       103870 non-null  object 
 17  房屋年限       59361 non-null   object 
 18  产权所属       103871 non-null  object 
 19  lon        103871 non-n

In [6]:
price_train['交易权属'].unique()

array(['商品房', '已购公房', '限价商品房', '一类经济适用房', '央产房', '私产', '二类经济适用房', '定向安置房',
       '房改房', '经济适用房', '拆迁还建房', '集资房', '动迁安置房', '售后公房', '回迁房'],
      dtype=object)

In [7]:
price_test.describe()

,城市,区域,板块,lon,lat,年份,容积率,停车位
count,34017.000000,34017.000000,34017.000000,34017.000000,34017.000000,34017.000000,24714.000000,24446.000000
mean,3.827351,66.829850,603.498751,115.808059,33.030666,2022.000764,2.612771,1114.219831
std,3.309469,35.773038,345.584708,5.677953,6.131005,0.027636,1.552519,1388.954171
min,0.000000,1.000000,1.000000,103.482624,23.025900,2022.000000,0.020000,1.000000
25%,1.000000,35.000000,303.000000,114.290508,30.481652,2022.000000,1.730000,255.000000
50%,3.000000,68.000000,584.000000,117.346794,32.268684,2022.000000,2.300000,650.000000
75%,6.000000,92.000000,924.000000,121.660158,40.755887,2022.000000,3.000000,1495.000000
max,11.000000,131.000000,1186.000000,122.958938,42.693676,2023.000000,35.000000,8700.000000


In [8]:
price_train.loc[:, '产权所属'] = price_train['产权所属'].map({'共有': 1, '非共有': 0})
price_test.loc[:, '产权所属'] = price_test['产权所属'].map({'共有': 1, '非共有': 0})

In [9]:
def extract_fee_value(text):
    """提取费用数值并计算平均值（适用于物业费、燃气费、供热费等）"""
    if pd.isna(text):
        return None
    
    numbers = re.findall(r'\d+\.?\d*', str(text).replace(' ', ''))
    if numbers:
        return sum(map(float, numbers)) / len(numbers)
    return np.nan

def extract_num(df, columns=['建筑面积','套内面积','房屋总数','楼栋总数','物业费','燃气费','供热费','绿化率']):
    for col in columns:
        if col == '建筑面积':
            df[col] = df[col].astype(str).str.extract(r'(\d+\.?\d*)㎡')
            df[col] = pd.to_numeric(df[col], errors='coerce')
        elif col == '套内面积':
            df[col] = df[col].astype(str).str.extract(r'(\d+\.?\d*)㎡')
            df[col] = pd.to_numeric(df[col], errors='coerce')
        elif col == '房屋总数':
            df[col] = df[col].astype(str).str.extract(r'(\d+\.?\d*)户')
            df[col] = pd.to_numeric(df[col], errors='coerce')
        elif col == '楼栋总数':
            df[col] = df[col].astype(str).str.extract(r'(\d+\.?\d*)栋')
            df[col] = pd.to_numeric(df[col], errors='coerce')
        elif col in ['物业费', '燃气费', '供热费']:
            # 使用统一的提取函数
            df[col] = df[col].apply(extract_fee_value)
        elif col == '绿化率':
            df[col] = df[col].astype(str).str.extract(r'(\d+(?:\.\d+)?)\s*%?')
            df[col] = pd.to_numeric(df[col], errors='coerce')
            
    return df

price_train=extract_num(price_train)
price_test=extract_num(price_test)

In [10]:
def House_Layout_Features(df):       
    # 房屋户型解析
    if "房屋户型" in df.columns:
        def parse_layout(text):
            text = str(text)
            rooms = re.findall(r'(\d+)室', text)
            halls = re.findall(r'(\d+)厅', text)
            kitchens = re.findall(r'(\d+)厨', text)
            baths = re.findall(r'(\d+)卫', text)
            return pd.Series([
                int(rooms[0]) if rooms else 0,
                int(halls[0]) if halls else 0,
                int(kitchens[0]) if kitchens else 0,
                int(baths[0]) if baths else 0
            ])
        df[["卧室数","客厅数","厨房数","卫生间数"]] = df["房屋户型"].apply(parse_layout)

    # 创建"是否别墅"特征
    if "别墅类型" in df.columns:
        df["是否别墅"] = df["别墅类型"].notna().astype(int)
    else:
        df["是否别墅"] = 0  # 若无该列，则默认为普通住宅


    # 楼层信息解析
    df = parse_floor_info(df)
    
    # 梯户比例解析
    if "梯户比例" in df.columns:
        df["梯户比例"] = df.apply(parse_lift_ratio, axis=1)
    
    # 电梯信息推断
    df = infer_elevator_by_building_type(df)
    
    return df


def parse_floor_info(df):
    """楼层信息解析函数"""
    
    def parse_single_floor(floor_text):
        """解析单个楼层文本"""
        text = str(floor_text)
        
        # 提取总楼层数
        total_match = re.search(r'共(\d+)层', text)
        total_floor = int(total_match.group(1)) if total_match else None
        
        # 定义楼层类型映射
        floor_type_mapping = {
            '地下室': 'basement',
            '底层': 'ground', 
            '低楼层': 'low',
            '中楼层': 'middle',
            '高楼层': 'high',
            '顶层': 'top'
        }
        
        # 识别楼层类型
        floor_type = None
        for key, value in floor_type_mapping.items():
            if key in text:
                floor_type = value
                break
        
        # 计算当前楼层
        current_floor = None
        if total_floor and floor_type:
            if floor_type == 'basement':
                current_floor = -1
            elif floor_type == 'ground':
                current_floor = 1
            elif floor_type == 'low':
                current_floor = max(2, round(total_floor * 0.25))
            elif floor_type == 'middle':
                current_floor = round(total_floor * 0.5)
            elif floor_type == 'high':
                current_floor = round(total_floor * 0.75)
            elif floor_type == 'top':
                current_floor = total_floor
        
        return pd.Series([current_floor, total_floor, floor_type])
    
    # 应用楼层解析
    df[['当前楼层', '总楼层', '楼层类型']] = df['所在楼层'].apply(parse_single_floor)
    
    # 创建额外的楼层特征
    df['相对楼层位置'] = df['当前楼层'] / df['总楼层']
    df['是否为底层'] = (df['楼层类型'] == 'ground').astype(int)
    df['是否为顶层'] = (df['楼层类型'] == 'top').astype(int)
    df['是否为地下室'] = (df['楼层类型'] == 'basement').astype(int)
    
    # 建筑高度分类
    def classify_building_height(total_floor):
        if pd.isna(total_floor):
            return '0'
        elif total_floor <= 6:
            return '1'      # 低层
        elif total_floor <= 11:
            return '2'      # 小高层
        elif total_floor <= 18:
            return '3'      # 高层
        else:
            return '4'      # 超高层
    
    df['建筑类型'] = df['总楼层'].apply(classify_building_height)
    
    # 别墅特殊处理
    df = process_villa_floors(df)
    
    # 处理异常值
    df = handle_floor_outliers(df)
    
    return df


def process_villa_floors(df):
    """别墅楼层特殊处理"""
    if '别墅类型' in df.columns:
        villa_mask = df['是否别墅'] == 1
        
        # 对于别墅，楼层信息意义不同
        df.loc[villa_mask, '楼层类型'] = 'villa'
        df.loc[villa_mask, '相对楼层位置'] = 0.5
        df.loc[villa_mask, '总楼层'] = 3  # 别墅通常3层
    
    return df


def handle_floor_outliers(df):
    """处理楼层异常值"""
    # 当前楼层不能大于总楼层
    invalid_mask = (df['当前楼层'] > df['总楼层']) & df['当前楼层'].notna() & df['总楼层'].notna()
    df.loc[invalid_mask, '当前楼层'] = df.loc[invalid_mask, '总楼层']
    
    # 地下室特殊处理
    basement_mask = df['楼层类型'] == 'basement'
    df.loc[basement_mask, '相对楼层位置'] = -0.1
    
    return df


def parse_lift_ratio(row):
    """解析梯户比例"""
    text = str(row.get("梯户比例", ""))
    villa_type = str(row.get("别墅类型", ""))
    is_villa = row.get("是否别墅", 0)
    
    # 中文数字解析函数
    def cn2num(s):
        if not s:
            return np.nan
        if s.isdigit():
            return int(s)
        num_map = {
            '零':0, '一':1, '二':2, '两':2, '三':3, '四':4, '五':5,
            '六':6, '七':7, '八':8, '九':9
        }
        total = 0
        if '十' in s:
            parts = s.split('十')
            if parts[0] == '':
                total += 10
            else:
                total += num_map.get(parts[0], 0) * 10
            if len(parts) > 1 and parts[1] != '':
                total += num_map.get(parts[1], 0)
        else:
            total += num_map.get(s, np.nan)
        return total
    
    result = np.nan
    
    # 普通住宅
    if is_villa == 0:
        pattern = re.findall(
            r'([一二两三四五六七八九十\d]{1,3})梯[^\d一二两三四五六七八九十]{0,2}([一二两三四五六七八九十\d]{1,3})户', text
        )
        if pattern:
            t_raw, h_raw = pattern[0]
            t_val, h_val = cn2num(t_raw), cn2num(h_raw)
            if pd.notna(t_val) and pd.notna(h_val) and h_val != 0:
                return t_val / h_val
    
    # 别墅类
    else:
        if "独栋" in villa_type: 
            result = 1.0
        elif "双拼" in villa_type: 
            result = 0.75
        elif "联排" in villa_type: 
            result = 0.5
        elif "叠拼" in villa_type: 
            result = 0.33
        else:
            result = 0.5
    
    return result


def infer_elevator_by_building_type(df):
    """根据建筑类型推断电梯配备情况"""
    if '配备电梯' in df.columns:
        # 先将现有数据映射
        df['配备电梯'] = df['配备电梯'].map({'有': 1, '无': 0})
        
        # 根据建筑类型推断缺失的电梯信息
        high_rise_mask = df['建筑类型'].isin(['2', '3', '4']) & df['配备电梯'].isna()
        df.loc[high_rise_mask, '配备电梯'] = 1
        
        low_rise_mask = (df['建筑类型'] == '1') & df['配备电梯'].isna()
        df.loc[low_rise_mask, '配备电梯'] = 0
        
        unknown_mask = (df['建筑类型'] == '0') & df['配备电梯'].isna()
        df.loc[unknown_mask, '配备电梯'] = 0
        
        # 确保所有值为数值类型
        df['配备电梯'] = df['配备电梯'].fillna(0).astype(int)
    
    return df


# 使用示例
price_train = House_Layout_Features(price_train)
price_test = House_Layout_Features(price_test)

In [11]:
# 停车费用处理
# 1. 通用金额→月费换算函数
def convert_to_month_fee(value, unit_hint=None):
    """根据金额及上下文提示换算为月费（元/月）"""
    if value is None or pd.isna(value):
        return np.nan
    value = float(value)
    
    # 单位提示优先
    if unit_hint == 'hour':
        return value * 240
    elif unit_hint == 'day':
        return value * 30
    elif unit_hint == 'year':
        return value / 12
    elif unit_hint == 'month':
        return value
    
    # 没有单位提示 -> 通过数值范围推断
    if value == 0:
        return 0
    elif value < 20:        # 太小 -> 按小时算
        return value * 240
    elif value < 1000:     # 正常区间 -> 月费
        return value
    elif value >= 1000:    # 超大值 -> 售价，忽略
        return np.nan
    
    return np.nan
# 2 中文数字单位转化
def chinese_number_to_float(text):
    """
    将带有中文单位的数字（如 '1万', '2千', '3百'）转换为浮点数。
    不带单位的数字直接转 float。
    """
    if text is None or text == '':
        return np.nan

    text = str(text).strip()
    
    # 去除“元”、“块”、“/月”等干扰字符
    t = re.sub(r'[元块\/每月位]+', '', text)
    
    # 处理带单位的情况
    if re.search(r'万', t):
        base = float(re.sub(r'万.*', '', t))
        return base * 10000
    elif re.search(r'千', t):
        base = float(re.sub(r'千.*', '', t))
        return base * 1000
    elif re.search(r'百', t):
        base = float(re.sub(r'百.*', '', t))
        return base * 100
    else:
        # 无单位时，取第一个数字
        match = re.search(r'\d+\.?\d*', t)
        if match:
            return float(match.group())
        else:
            return np.nan
            
# 3 从文本中提取所有相关费用的函数
def extract_values(text):
    """解析原始文本，返回提取出的多种类型价格"""
    if not isinstance(text, str) or text.strip() == '':
        return {}
    
    t = text.replace('～','-').replace('~','-').replace('；',',').replace('。',',').replace('、',',')

    #将中文转化为数字
    for m in re.finditer(r'([\d\.]+(?:万|千|百)?)[元块]*/?(小时|时|天|月|年)?', t):
        val_raw, unit = m.groups()
        val = chinese_number_to_float(val_raw)
    
    # 免费
    if re.search(r'免费|不收费|0元', t):
        return {'free': True}
    
    result = {
        'ground': [], 'underground': [], 'outdoor': [], 'indoor': [],
        'fixed': [], 'unfixed': [],
        'owner': [], 'tenant': [],
        'hour': [], 'day': [], 'month': [], 'year': []
    }
    
    # 区间价（如300-500）
    ranges = re.findall(r'(\d+\.?\d*)[-~～](\d+\.?\d*)', t)
    for a, b in ranges:
        result['month'].append((float(a)+float(b))/2)
    
    # 单价识别（带单位）
    for m in re.finditer(r'(\d+\.?\d*)元?/?(小时|时|天|月|年)?', t):
        val, unit = m.groups()
        if not val:
            continue
        val = float(val)
        unit_map = {'小时':'hour','时':'hour','天':'day','月':'month','年':'year'}
        unit = unit_map.get(unit, None)
        
        # 归类主体
        segment = t[max(0, m.start()-6):m.end()+6]
        if re.search(r'地上', segment):
            result['ground'].append(convert_to_month_fee(val, unit))
        elif re.search(r'地下', segment):
            result['underground'].append(convert_to_month_fee(val, unit))
        elif re.search(r'露天', segment):
            result['outdoor'].append(convert_to_month_fee(val, unit))
        elif re.search(r'室内', segment):
            result['indoor'].append(convert_to_month_fee(val, unit))
        elif re.search(r'固定', segment) and not re.search(r'不固定', segment):
            result['fixed'].append(convert_to_month_fee(val, unit))
        elif re.search(r'不固定', segment):
            result['unfixed'].append(convert_to_month_fee(val, unit))
        elif re.search(r'业主', segment):
            result['owner'].append(convert_to_month_fee(val, unit))
        elif re.search(r'租客', segment):
            result['tenant'].append(convert_to_month_fee(val, unit))
        elif re.search(r'临保|小时|时', segment):
            result['hour'].append(convert_to_month_fee(val, unit))
        else:
            result['month'].append(convert_to_month_fee(val, unit))
    
    return result

# 3️ 主函数：从多类型结果中统一为一个月度费用
def unify_month_fee(text):
    info = extract_values(text)
    
    if info.get('free'):
        return 0.0
    
    # 地上/地下/露天/室内 → 环境类取均值
    env_fees = []
    for key in ['ground','underground','outdoor','indoor']:
        vals = [v for v in info.get(key, []) if v is not None and not pd.isna(v)]
        if len(vals)>0:
            env_fees.extend(vals)
    if len(env_fees)==1:
        env_fee = env_fees[0]
    elif len(env_fees)>=2:
        # 如果包含0，则取非0均值
        non_zero = [v for v in env_fees if v>0]
        env_fee = np.mean(non_zero) if len(non_zero)>0 else 0
    else:
        env_fee = np.nan
    
    # 固定/不固定取均值
    fix_vals = [v for v in info.get('fixed', []) + info.get('unfixed', []) if not pd.isna(v)]
    fix_fee = np.mean(fix_vals) if len(fix_vals)>0 else np.nan
    
    # 业主/租客取均值
    own_vals = [v for v in info.get('owner', []) + info.get('tenant', []) if not pd.isna(v)]
    own_fee = np.mean(own_vals) if len(own_vals)>0 else np.nan
    
    # 临保/小时取均值
    hour_vals = [v for v in info.get('hour', []) if not pd.isna(v)]
    hour_fee = np.mean(hour_vals) if len(hour_vals)>0 else np.nan
    
    # 普通月费
    month_vals = [v for v in info.get('month', []) if not pd.isna(v)]
    month_fee = np.mean(month_vals) if len(month_vals)>0 else np.nan
    
    # 综合选择优先级：环境类 > 固定类 > 业主类 > 月保 > 小时类
    candidates = [env_fee, fix_fee, own_fee, month_fee, hour_fee]
    candidates = [v for v in candidates if not pd.isna(v)]
    
    if len(candidates)==0:
        return np.nan
    else:
        return round(float(np.mean(candidates)), 2)

# 4️ 应用并输出结果

price_train['停车费用'] = price_train['停车费用'].apply(unify_month_fee)
price_test['停车费用'] = price_test['停车费用'].apply(unify_month_fee)

In [12]:
def house_relevant_time(df):    
    df = df.copy()
    
    if df['年份'].dtype == 'object':
        # 如果是字符串，提取数字
        df['年份'] = df['年份'].str.extract(r'(\d+)').astype(float)
    
    # 检查年份列的当前类型
    if pd.api.types.is_datetime64_any_dtype(df['年份']):
        # 如果已经是datetime类型，不需要转换
        pass
    else:
        # 确保年份是整数，然后转换为字符串，再转换为日期
        # 使用安全的转换方法
        df['年份'] = df['年份'].fillna(0).astype('int64').astype(str) + '-01-01'
        df['年份'] = pd.to_datetime(df['年份'], errors='coerce')
    df['交易时间'] = pd.to_datetime(df['交易时间'], errors='coerce')
    df['上次交易'] = pd.to_datetime(df['上次交易'], errors='coerce')
    df['交易到登记周期']=(df['年份']-df['上次交易']).dt.days / 365.25
    df['交易周期']=(df['交易时间']-df['上次交易']).dt.days /365.25
    return df

price_train=house_relevant_time(price_train)
price_test=house_relevant_time(price_test)

In [13]:
def extract_years(text):
    """从字符串中提取起始年份和结束年份"""
    years = re.findall(r'\d{4}', str(text))
    
    if not years:
        return np.nan, np.nan
    elif len(years) == 1:
        return int(years[0]), int(years[0])  # 单一年份
    else:
        return int(years[0]), int(years[1])  # 年份区间
def process_construction_period(df, column_name='建筑年代'):
    """处理建筑年代区间数据"""
    df = df.copy()
    
    # 提取年份并计算相关特征
    df[['建筑起始年份', '建筑结束年份']] = df[column_name].apply(
        lambda x: pd.Series(extract_years(x)) if not pd.isna(x) else pd.Series([np.nan, np.nan])
    )
    
    # 计算建筑特征
    df['平均建筑年份'] = (df['建筑起始年份'] + df['建筑结束年份']) / 2
    
    # 创建分组和质量特征
    trade_year = df['交易时间'].dt.year
    
    # 房龄特征
    df['小区最新房龄'] = trade_year - df['建筑结束年份']
    df['小区最老房龄'] = trade_year - df['建筑起始年份']
    df['小区平均房龄'] = (df['小区最新房龄'] + df['小区最老房龄']) / 2
    df['小区房龄差异'] = df['小区最老房龄'] - df['小区最新房龄']
    
    # 政策周期编号
    policy_conditions = [
        df['建筑结束年份'] < 1998,
        (df['建筑起始年份'] >= 1998) & (df['建筑结束年份'] <= 2007),
        (df['建筑起始年份'] >= 2008) & (df['建筑结束年份'] <= 2010),
        (df['建筑起始年份'] >= 2011) & (df['建筑结束年份'] <= 2014),
        (df['建筑起始年份'] >= 2015) & (df['建筑结束年份'] <= 2016),
        df['建筑起始年份'] >= 2017
    ]
    
    policy_periods = ['房改前', '黄金十年', '四万亿', '限购期', '去库存', '房住不炒']

    df['政策周期'] = np.select(policy_conditions, policy_periods, default='未知')

    df = df.drop(['建筑起始年份', '建筑结束年份','小区最新房龄','小区最老房龄','平均建筑年份'], axis=1)
    return df

price_train=process_construction_period(price_train)
price_test=process_construction_period(price_test)

In [14]:
num_cols = price_train.select_dtypes(include=[np.number]).columns.tolist()

In [15]:
def safe_float_conversion(value, default=0.0):
    """
    安全地将值转换为浮点数
    """
    try:
        return float(value)
    except (ValueError, TypeError):
        return default

def haversine_distance(lat1, lon1, lat2, lon2):
    """
    计算两个经纬度坐标之间的球面距离（公里）
    """
    # 安全地转换为浮点数
    lat1 = safe_float_conversion(lat1)
    lon1 = safe_float_conversion(lon1)
    lat2 = safe_float_conversion(lat2)
    lon2 = safe_float_conversion(lon2)
    
    # 检查是否有无效的坐标值
    if any(pd.isna([lat1, lon1, lat2, lon2])):
        return 0
    
    R = 6371  # 地球半径，单位：公里
    
    try:
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
        c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
        
        return R * c
    except Exception as e:
        print(f"距离计算错误: {e}, 坐标: ({lat1}, {lon1}), ({lat2}, {lon2})")
        return 0

def calculate_city_centers(df, city_col='城市', lon_col='lon', lat_col='lat', method='convex_hull'):
    """
    自动计算每个城市的中心点
    """
    city_centers = {}
    
    # 确保我们只使用有效的经纬度列
    required_cols = [city_col, lon_col, lat_col]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"数据中缺少必要的列: {missing_cols}")
    
    # 按城市分组
    grouped = df.groupby(city_col)
    
    for city_id, group in grouped:
        # 确保经纬度是数值类型
        group = group.copy()
        group[lon_col] = group[lon_col].apply(safe_float_conversion)
        group[lat_col] = group[lat_col].apply(safe_float_conversion)
        
        # 移除无效的坐标
        valid_coords = group[(group[lon_col] != 0) & (group[lat_col] != 0)]
        
        if len(valid_coords) < 1:
            print(f"警告: 城市 {city_id} 没有有效的坐标数据")
            continue
            
        if len(valid_coords) < 3:  # 如果数据点太少，使用简单平均
            center_lon = valid_coords[lon_col].mean()
            center_lat = valid_coords[lat_col].mean()
        else:
            if method == 'mean':
                center_lon = valid_coords[lon_col].mean()
                center_lat = valid_coords[lat_col].mean()
            elif method == 'median':
                center_lon = valid_coords[lon_col].median()
                center_lat = valid_coords[lat_col].median()
            elif method == 'convex_hull':
                # 使用凸包中心作为城市中心
                try:
                    points = valid_coords[[lon_col, lat_col]].values
                    hull = ConvexHull(points)
                    # 计算凸包顶点的中心
                    hull_points = points[hull.vertices]
                    center_lon = hull_points[:, 0].mean()
                    center_lat = hull_points[:, 1].mean()
                except Exception as e:
                    print(f"城市 {city_id} 凸包计算失败: {e}，使用中位数代替")
                    center_lon = valid_coords[lon_col].median()
                    center_lat = valid_coords[lat_col].median()
            else:
                raise ValueError(f"不支持的method: {method}")
        
        city_centers[city_id] = (center_lon, center_lat)
    
    print(f"计算了 {len(city_centers)} 个城市的中心点")
    return city_centers

def calculate_ring_thresholds(df, city_col='城市', ring_col='环线', 
                             lon_col='lon', lat_col='lat', city_centers=None):
    """
    自动计算每个城市的环线距离阈值
    """
    if city_centers is None:
        city_centers = calculate_city_centers(df, city_col, lon_col, lat_col)
    
    ring_definitions = {}
    
    # 按城市分组
    grouped = df.groupby(city_col)
    
    for city_id, group in grouped:
        if city_id not in city_centers:
            continue
            
        center_lon, center_lat = city_centers[city_id]
        
        # 计算每个点到城市中心的距离
        distances = []
        rings = []
        for _, row in group.iterrows():
            if pd.notna(row[ring_col]) and row[ring_col] != '':
                # 确保经纬度是数值类型
                lat_val = safe_float_conversion(row[lat_col])
                lon_val = safe_float_conversion(row[lon_col])
                
                # 跳过无效坐标
                if lat_val == 0 and lon_val == 0:
                    continue
                    
                distance = haversine_distance(
                    lat_val, lon_val, center_lat, center_lon
                )
                distances.append(distance)
                rings.append(row[ring_col])
        
        if not distances:
            continue
        
        # 创建距离和环线的对应关系
        ring_distance_pairs = list(zip(rings, distances))
        
        # 按环线分组，计算每个环线的平均距离
        ring_stats = {}
        for ring in set(rings):
            ring_distances = [d for r, d in ring_distance_pairs if r == ring]
            if ring_distances:
                ring_stats[ring] = {
                    'mean': np.mean(ring_distances),
                    'median': np.median(ring_distances),
                    'min': np.min(ring_distances),
                    'max': np.max(ring_distances),
                    'count': len(ring_distances)
                }
        
        # 确定环线距离阈值（使用最大值）
        ring_thresholds = {}
        sorted_rings = sorted(ring_stats.keys(), 
                             key=lambda x: ring_stats[x]['mean'])
        
        for i, ring in enumerate(sorted_rings):
            if i < len(sorted_rings) - 1:
                # 当前环线的最大距离作为阈值
                ring_thresholds[ring] = ring_stats[ring]['max']
            else:
                # 最后一个环线使用一个较大的值
                ring_thresholds[ring] = ring_stats[ring]['max'] * 1.5
        
        ring_definitions[city_id] = {
            'center': (center_lon, center_lat),
            'rings': ring_thresholds
        }
    
    print(f"为 {len(ring_definitions)} 个城市计算了环线阈值")
    return ring_definitions



In [16]:
class AutoCenterRingLinePredictor:
    """
    自动计算城市中心的环线预测器
    """
    
    def __init__(self):
        self.model = None
        self.features = []
        self.encoders = {}
        self.scalers = {}
        self.city_centers = None
        self.ring_definitions = None
        self.is_trained = False
    
    def auto_define_city_areas(self, df, city_col='城市', ring_col='环线',
                              lon_col='lon', lat_col='lat'):
        """
        自动定义城市中心和环线区域
        """
        print("自动计算城市中心和环线区域...")
        
        # 1. 计算城市中心
        self.city_centers = calculate_city_centers(
            df, city_col, lon_col, lat_col, method='convex_hull'
        )
        
        # 2. 计算环线阈值（需要有环线标签的数据）
        if ring_col in df.columns and df[ring_col].notna().any():
            self.ring_definitions = calculate_ring_thresholds(
                df, city_col, ring_col, lon_col, lat_col, self.city_centers
            )
        else:
            print("警告: 没有环线标签数据，无法自动计算环线阈值")
            self.ring_definitions = {}
            
            # 为每个城市设置默认环线定义
            for city_id, center in self.city_centers.items():
                self.ring_definitions[city_id] = {
                    'center': center,
                    'rings': {
                        '一环': 3,
                        '二环': 6,
                        '三环': 10,
                        '四环': 15,
                        '五环': 20,
                        '五环外': 999
                    }
                }
        
        return self.ring_definitions
    
    def calculate_distance(self, lon1, lat1, lon2, lat2):
        """
        计算两个经纬度坐标之间的距离（公里）
        """
        # 使用统一的haversine_distance函数
        return haversine_distance(lat1, lon1, lat2, lon2)
    
    def _prepare_data(self, df):
        """
        数据预处理：确保数据类型正确并清理无效数据
        """
        df_processed = df.copy()
        
        # 打印列信息以便调试
        print(f"数据列: {list(df_processed.columns)}")
        
        # 确保经纬度是数值类型
        if 'lon' in df_processed.columns:
            df_processed['lon'] = df_processed['lon'].apply(safe_float_conversion)
        else:
            raise ValueError("数据中缺少'lon'列")
            
        if 'lat' in df_processed.columns:
            df_processed['lat'] = df_processed['lat'].apply(safe_float_conversion)
        else:
            raise ValueError("数据中缺少'lat'列")
        
        # 删除包含无效经纬度的行
        invalid_coords = df_processed[(df_processed['lon'] == 0) & (df_processed['lat'] == 0)]
        if len(invalid_coords) > 0:
            print(f"警告: 发现 {len(invalid_coords)} 行无效的经纬度数据，将被删除")
            df_processed = df_processed[~((df_processed['lon'] == 0) & (df_processed['lat'] == 0))]
        
        # 检查并清理城市列
        if '城市' not in df_processed.columns:
            raise ValueError("数据中缺少'城市'列")
        
        # 删除城市为空的记录
        city_missing = df_processed['城市'].isna().sum()
        if city_missing > 0:
            print(f"警告: 发现 {city_missing} 行缺失城市数据，将被删除")
            df_processed = df_processed.dropna(subset=['城市'])
        
        return df_processed
    
    def _calculate_distances_to_center(self, df_processed):
        """
        计算每个点到对应城市中心的距离
        """
        distances = []
        for _, row in df_processed.iterrows():
            try:
                city_id = row['城市']
                if city_id in self.city_centers:
                    center_lon, center_lat = self.city_centers[city_id]
                    distance = self.calculate_distance(
                        row['lon'], row['lat'], center_lon, center_lat
                    )
                else:
                    # 未知城市，使用所有城市的平均距离
                    if self.city_centers:
                        all_distances = []
                        for center_lon, center_lat in self.city_centers.values():
                            dist = self.calculate_distance(
                                row['lon'], row['lat'], center_lon, center_lat
                            )
                            all_distances.append(dist)
                        distance = np.mean(all_distances) if all_distances else 0
                    else:
                        distance = 0
                distances.append(distance)
            except Exception as e:
                print(f"计算距离时出错: {e}")
                distances.append(0)
        
        return distances
    
    def create_features(self, df, is_training=True):
        """
        创建特征矩阵
        """
        # 数据预处理
        df_processed = self._prepare_data(df)
        
        # 1. 基础地理特征
        features = ['lon', 'lat']
        
        # 2. 距城市中心距离
        if self.city_centers is None:
            self.auto_define_city_areas(df_processed)
        
        df_processed['距中心距离'] = self._calculate_distances_to_center(df_processed)
        features.append('距中心距离')
        
        # 3. 标准化数值特征
        numerical_features = ['lon', 'lat', '距中心距离']
        if is_training:
            self.scalers['numerical'] = StandardScaler()
            df_processed[numerical_features] = self.scalers['numerical'].fit_transform(
                df_processed[numerical_features]
            )
        else:
            df_processed[numerical_features] = self.scalers['numerical'].transform(
                df_processed[numerical_features]
            )
        
        self.features = features
        return df_processed, features
    
    def train_with_missing_rings(self, df_train, target_col='环线', cv_folds=5):
        """
        在训练集上训练模型，处理训练集中的环线缺失
        """
        print("开始训练环线预测模型（处理训练集环线缺失）...")
        
        # 1. 计算城市中心（使用所有训练数据）
        self.city_centers = self.auto_define_city_areas(df_train)
        
        # 2. 分离有环线和无环线的训练数据
        df_train_with_rings = df_train[df_train[target_col].notna() & (df_train[target_col] != '')].copy()
        df_train_missing_rings = df_train[df_train[target_col].isna() | (df_train[target_col] == '')].copy()
        
        print(f"训练集有环线数据: {len(df_train_with_rings)} 条")
        print(f"训练集缺失环线数据: {len(df_train_missing_rings)} 条")
        
        if len(df_train_with_rings) == 0:
            raise ValueError("训练集中没有可用的环线标签数据")
        
        # 3. 准备有环线数据的特征
        df_train_processed, features = self.create_features(df_train_with_rings, is_training=True)
        
        # 4. 分离特征和目标变量
        X_train = df_train_processed[features].values
        y_train = df_train_processed[target_col].values
        
        print(f"有效训练数据: {X_train.shape[0]}")
        print(f"特征数量: {X_train.shape[1]}")
        print(f"环线: {np.unique(y_train)}")
        
        # 5. 训练随机森林模型
        self.model = RandomForestClassifier(
            n_estimators=100,
            max_depth=15,
            min_samples_split=5,
            min_samples_leaf=2,
            random_state=42,
            class_weight='balanced',
            n_jobs=-1
        )
        
        self.model.fit(X_train, y_train)
        
        # 6. 交叉验证评估
        cv_scores = cross_val_score(self.model, X_train, y_train, cv=min(cv_folds, len(y_train)), scoring='accuracy')
        print(f"交叉验证准确率: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
        
        # 7. 训练集上的表现
        y_pred_train = self.model.predict(X_train)
        train_accuracy = accuracy_score(y_train, y_pred_train)
        print(f"训练集准确率: {train_accuracy:.4f}")
        
        # 8. 特征重要性
        feature_importance = pd.DataFrame({
            'feature': features,
            'importance': self.model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        print("\n特征重要性:")
        print(feature_importance)
        
        self.is_trained = True
        print("模型训练完成!")
        
        return self.model, features
    
    def fill_missing_rings(self, df, target_col='环线', return_proba=False):
        """
        填充数据集中的缺失环线
        """
        if not self.is_trained:
            raise ValueError("模型尚未训练，请先调用 train_with_missing_rings() 方法")
        
        print(f"开始填充缺失环线...")
        
        # 1. 准备数据（使用训练时的预处理）
        df_processed, _ = self.create_features(df, is_training=False)
        
        # 2. 提取特征
        X = df_processed[self.features].values
        
        # 3. 预测
        predictions = self.model.predict(X)
        
        # 4. 创建结果DataFrame
        df_result = df.copy()
        
        # 5. 只填充缺失的环线
        mask_missing = df_result[target_col].isna() | (df_result[target_col] == '')
        df_result.loc[mask_missing, target_col] = predictions[mask_missing]
        
        print(f"填充了 {mask_missing.sum()} 条缺失环线记录")
        
        # 6. 预测概率（可选）
        if return_proba:
            probabilities = self.model.predict_proba(X)
            confidence = np.max(probabilities, axis=1)
            df_result['环线预测置信度'] = confidence
            
            # 为每个类别保存概率
            classes = self.model.classes_
            for i, class_name in enumerate(classes):
                df_result[f'{class_name}_概率'] = probabilities[:, i]
        
        return df_result



In [17]:
def complete_ring_filling_workflow(price_train, price_test, target_col='环线'):
    """
    完整的环线填充工作流程 - 处理训练集和测试集的环线缺失
    """
    price_train = price_train.copy()
    price_test = price_test.copy()

    print("=== 环线填充工作流程开始 ===")
    print(f"训练集原始大小: {len(price_train)}")
    print(f"测试集原始大小: {len(price_test)}")
    
    # 1. 初始化预测器
    predictor = AutoCenterRingLinePredictor()
    
    # 2. 分析数据情况
    train_missing = price_train[target_col].isna().sum() + (price_train[target_col] == '').sum()
    test_missing = price_test[target_col].isna().sum() + (price_test[target_col] == '').sum()
    
    print(f"训练集环线缺失: {train_missing} 条 ({train_missing/len(price_train)*100:.1f}%)")
    print(f"测试集环线缺失: {test_missing} 条 ({test_missing/len(price_test)*100:.1f}%)")
    print(f"训练集有环线数据: {len(price_train) - train_missing} 条")
    
    # 3. 在训练集上训练模型（只使用有环线的数据）
    model, features = predictor.train_with_missing_rings(price_train, target_col=target_col)
    
    # 4. 填充训练集的缺失环线
    print("\n=== 填充训练集缺失环线 ===")
    price_train_filled = predictor.fill_missing_rings(price_train, target_col=target_col, return_proba=True)
    
    # 5. 填充测试集的缺失环线
    print("\n=== 填充测试集缺失环线 ===")
    price_test_filled = predictor.fill_missing_rings(price_test, target_col=target_col, return_proba=True)

    return price_train_filled, price_test_filled

In [18]:
price_train_filled, price_test_filled=complete_ring_filling_workflow(price_train, price_test, target_col='环线')

=== 环线填充工作流程开始 ===
训练集原始大小: 103871
测试集原始大小: 34017
训练集环线缺失: 63452 条 (61.1%)
测试集环线缺失: 18340 条 (53.9%)
训练集有环线数据: 40419 条
开始训练环线预测模型（处理训练集环线缺失）...
自动计算城市中心和环线区域...
计算了 12 个城市的中心点
为 5 个城市计算了环线阈值
训练集有环线数据: 40419 条
训练集缺失环线数据: 63452 条
数据列: ['城市', '区域', '板块', '环线', '房屋户型', '所在楼层', '建筑面积', '套内面积', '建筑结构', '装修情况', '梯户比例', '配备电梯', '别墅类型', '交易时间', '交易权属', '上次交易', '房屋用途', '房屋年限', '产权所属', 'lon', 'lat', '年份', '建筑年代', '开发商', '房屋总数', '楼栋总数', '物业公司', '绿化率', '容积率', '物业费', '建筑结构_comm', '供水', '供暖', '供电', '燃气费', '供热费', '停车位', '停车费用', '卧室数', '客厅数', '厨房数', '卫生间数', '是否别墅', '当前楼层', '总楼层', '楼层类型', '相对楼层位置', '是否为底层', '是否为顶层', '是否为地下室', '建筑类型', '交易到登记周期', '交易周期', '小区平均房龄', '小区房龄差异', '政策周期']
有效训练数据: 40419
特征数量: 3
环线: ['三至四环' '中环至外环' '二环内' '二至三环' '五至六环' '六环外' '内环内' '内环至中环' '内环至外环' '四至五环'
 '外环外']
交叉验证准确率: 0.9420 (+/- 0.1831)
训练集准确率: 0.9986

特征重要性:
  feature  importance
1     lat    0.411571
0     lon    0.306146
2   距中心距离    0.282283
模型训练完成!

=== 填充训练集缺失环线 ===
开始填充缺失环线...
数据列: ['城市', '区域', '板块', '环线', '房屋户型', '所在楼层', 

In [19]:
columns_to_keep = [col for col in price_train.columns if '置信度' not in col and '概率' not in col]
price_train=price_train_filled[columns_to_keep]

In [20]:
columns_to_keep = [col for col in price_test.columns if '置信度' not in col and '概率' not in col]
price_test=price_test_filled[columns_to_keep]

In [21]:
class HierarchicalGeoClusterMapper:
    """
    分层地理聚类映射器，考虑城市、区域、板块层级
    """
    def __init__(self, eps=0.01, min_samples=5):
        self.eps = eps
        self.min_samples = min_samples
        self.city_models = {}  # 存储每个城市的聚类模型
        self.region_models = {}  # 存储每个区域的聚类模型
        self.is_fitted = False
    
    def fit(self, df):
        """
        在训练集上拟合分层聚类模型
        """
        results = []
        
        # 第一层：按城市分组
        for city in df['城市'].unique():
            city_data = df[df['城市'] == city].copy()
            
            # 存储城市级别的聚类模型
            city_model = {}
            
            # 第二层：按区域分组
            for region in city_data['区域'].unique():
                region_data = city_data[city_data['区域'] == region].copy()
                
                # 如果区域数据足够，进行聚类
                if len(region_data) >= self.min_samples:
                    # 使用经纬度进行聚类
                    coords = region_data[['lon', 'lat']].values
                    coords = np.nan_to_num(coords, nan=0.0)
                    
                    # DBSCAN聚类
                    dbscan = DBSCAN(eps=self.eps, min_samples=self.min_samples)
                    cluster_labels = dbscan.fit_predict(coords)
                    
                    # 处理噪声点
                    noise_mask = cluster_labels == -1
                    if noise_mask.any():
                        max_label = cluster_labels.max()
                        cluster_labels[noise_mask] = range(max_label + 1, max_label + 1 + noise_mask.sum())
                    
                    # 存储区域聚类模型
                    city_model[region] = {
                        'model': dbscan,
                        'labels': cluster_labels,
                        '聚类层级': '区域级'  # 明确设置聚类层级
                    }
                    
                    region_data.loc[:, '地理聚类'] = cluster_labels
                    region_data.loc[:, '聚类层级'] = '区域级'
                else:
                    # 样本太少，使用板块作为聚类
                    region_data.loc[:, '地理聚类'] = region_data['板块'].astype('category').cat.codes
                    region_data.loc[:, '聚类层级'] = '板块级'
                    
                    # 存储区域聚类信息
                    city_model[region] = {
                        '聚类层级': '板块级'  # 明确设置聚类层级
                    }
                
                results.append(region_data)
            
            self.city_models[city] = city_model
        
        self.is_fitted = True
        return pd.concat(results, ignore_index=True)
    
    def transform(self, df):
        """
        对测试集应用分层聚类模型
        """
        if not self.is_fitted:
            raise ValueError("必须先调用 fit 方法训练模型")
        
        results = []
        
        for city in df['城市'].unique():
            city_data = df[df['城市'] == city].copy()
            
            if city not in self.city_models:
                # 如果城市在训练集中不存在，使用默认聚类
                print(f"警告: 城市 '{city}' 在训练集中不存在，使用默认聚类")
                city_data.loc[:, '地理聚类'] = 0
                city_data.loc[:, '聚类层级'] = '默认级'
                results.append(city_data)
                continue
            
            city_model = self.city_models[city]
            
            for region in city_data['区域'].unique():
                region_data = city_data[city_data['区域'] == region].copy()
                
                if region not in city_model:
                    # 如果区域在训练集中不存在，使用默认聚类
                    print(f"警告: 区域 '{region}' 在训练集中不存在，使用默认聚类")
                    region_data.loc[:, '地理聚类'] = 0
                    region_data.loc[:, '聚类层级'] = '默认级'
                    results.append(region_data)
                    continue
                
                region_model_info = city_model[region]
                
                # 确保聚类层级键存在
                if '聚类层级' not in region_model_info:
                    print(f"警告: 区域 '{region}' 的聚类信息不完整，使用默认聚类")
                    region_data.loc[:, '地理聚类'] = 0
                    region_data.loc[:, '聚类层级'] = '默认级'
                    results.append(region_data)
                    continue
                
                if region_model_info['聚类层级'] == '区域级':
                    # 使用区域级聚类模型
                    coords = region_data[['lon', 'lat']].values
                    coords = np.nan_to_num(coords, nan=0.0)
                    
                    # 使用训练好的DBSCAN模型预测
                    cluster_labels = region_model_info['model'].fit_predict(coords)
                    
                    # 处理噪声点
                    noise_mask = cluster_labels == -1
                    if noise_mask.any():
                        max_label = cluster_labels.max()
                        cluster_labels[noise_mask] = range(max_label + 1, max_label + 1 + noise_mask.sum())
                    
                    region_data.loc[:, '地理聚类'] = cluster_labels
                    region_data.loc[:, '聚类层级'] = '区域级'
                elif region_model_info['聚类层级'] == '板块级':
                    # 使用板块级聚类
                    # 对于新板块，分配新的聚类标签
                    region_data.loc[:, '地理聚类'] = region_data['板块'].astype('category').cat.codes
                    region_data.loc[:, '聚类层级'] = '板块级'
                else:
                    # 未知的聚类层级，使用默认聚类
                    print(f"警告: 区域 '{region}' 的聚类层级未知，使用默认聚类")
                    region_data.loc[:, '地理聚类'] = 0
                    region_data.loc[:, '聚类层级'] = '默认级'
                
                results.append(region_data)
        
        return pd.concat(results, ignore_index=True)
    
    def fit_transform(self, df):
        """同时拟合和转换数据"""
        return self.fit(df)


def hierarchical_geo_clustering_with_mapping(train_df, test_df=None, eps=0.01, min_samples=5):
    """
    分层地理聚类，考虑城市、区域、板块层级
    
    参数:
    train_df: 训练集数据
    test_df: 测试集数据（可选）
    eps: DBSCAN的邻域半径
    min_samples: 形成核心点所需的最小样本数
    
    返回:
    如果只有训练集: 聚类后的训练集
    如果有训练集和测试集: (聚类后的训练集, 聚类后的测试集)
    """
    # 创建聚类器
    clusterer = HierarchicalGeoClusterMapper(eps=eps, min_samples=min_samples)
    
    # 在训练集上拟合模型
    train_clustered = clusterer.fit(train_df)
    
    if test_df is not None:
        # 对测试集应用模型
        test_clustered = clusterer.transform(test_df)
        return train_clustered, test_clustered
    else:
        return train_clustered


def main():
    # 方法1: 分层地理聚类
    train_clustered, test_clustered = hierarchical_geo_clustering_with_mapping(
        price_train, price_test, eps=0.01, min_samples=5
    )
    print("训练集聚类统计:")
    cluster_stats_train = train_clustered.groupby(['城市', '区域', '地理聚类']).size().reset_index(name='数量')
    print(cluster_stats_train.head(10))
    
    if test_clustered is not None:
        print("\n测试集聚类统计:")
        cluster_stats_test = test_clustered.groupby(['城市', '区域', '地理聚类']).size().reset_index(name='数量')
        print(cluster_stats_test.head(10))
    
    # 检查聚类一致性
    if test_clustered is not None:
        train_cities = set(train_clustered['城市'].unique())
        test_cities = set(test_clustered['城市'].unique())
        common_cities = train_cities.intersection(test_cities)
        
        print(f"\n城市覆盖分析:")
        print(f"训练集城市数: {len(train_cities)}")
        print(f"测试集城市数: {len(test_cities)}")
        print(f"共同城市数: {len(common_cities)}")
    
    return train_clustered, test_clustered


if __name__ == "__main__":
    train_result, test_result = main()

警告: 区域 '1.0' 在训练集中不存在，使用默认聚类
警告: 区域 '79.0' 在训练集中不存在，使用默认聚类
警告: 区域 '61.0' 在训练集中不存在，使用默认聚类
训练集聚类统计:
   城市   区域  地理聚类   数量
0   0  5.0     0  133
1   0  5.0     1   35
2   0  5.0     2   75
3   0  5.0     3   28
4   0  5.0     4   68
5   0  5.0     5    9
6   0  5.0     6   14
7   0  5.0     7    1
8   0  5.0     8    1
9   0  7.0     0   38

测试集聚类统计:
   城市   区域  地理聚类  数量
0   0  5.0     0  57
1   0  5.0     1  59
2   0  5.0     2  43
3   0  5.0     3  14
4   0  5.0     4  15
5   0  5.0     5   8
6   0  5.0     6   1
7   0  5.0     7   1
8   0  5.0     8   1
9   0  7.0     0  58

城市覆盖分析:
训练集城市数: 12
测试集城市数: 12
共同城市数: 12


In [22]:
price_train=train_result
price_test=test_result

In [23]:
class TextCategoryCleaner:
    def __init__(self):
        self.category_mappings = {}
        self.label_encoders = {}
        self.vectorizers = {}
        self.fitted = False
    
    def create_standard_categories(self):
        """定义标准化的分类体系"""
        return {
            '环线': {
                'standard_categories': ['内环内', '内环至中环', '中环至外环', '外环外', '二环内', '二至三环', '三至四环', '四至五环', '五至六环', '六环外', '未知'],
                'mapping_rules': {
                    '内环内': ['内环内'],
                    '内环至中环': ['内环至中环'],
                    '中环至外环': ['中环至外环'],
                    '外环外': ['外环外'],
                    '二环内': ['二环内'],
                    '二至三环': ['二至三环'],
                    '三至四环': ['三至四环'],
                    '四至五环': ['四至五环'],
                    '五至六环': ['五至六环'],
                    '六环外': ['六环外'],
                    '未知': ['未知', '其他']
                }
            },
            '聚类层级': {
                'standard_categories': ['默认级', '区域级', '板块级', '未知'],
                'mapping_rules': {
                    '默认级': ['默认级'],
                    '区域级': ['区域级'],
                    '板块级': ['板块级'],
                    '未知': ['未知', '其他']
                }
            },
            '建筑结构': {
                'standard_categories': ['砖混结构', '框架结构', '钢结构', '钢混结构', '砖木结构', '混合结构', '未知结构'],
                'mapping_rules': {
                    '混合结构': ['混合'],
                    '钢混结构': ['钢混', '钢筋混凝土'],
                    '框架结构': ['框架'],
                    '钢结构': ['钢构', '钢结构'],
                    '砖混结构': ['砖混'],
                    '砖木结构': ['砖木'],
                    '未知结构': ['未知', '其他', '不详']
                }
            },
            '供暖': {
                'standard_categories': ['集中供暖', '自采暖', '无供暖', '混合供暖'],
                'mapping_rules': {
                    '集中供暖': ['集中'],
                    '自采暖': ['自采', '自供暖'],
                    '无供暖': ['无', '没有'],
                    '混合供暖': ['混合', '集中/自采', '自采/集中']
                }
            },
            '建筑结构_comm': {
                'standard_categories': ['板楼', '塔楼', '板塔结合', '平房', '混合类型'],
                'mapping_rules': {
                    '板楼': ['板楼'],
                    '塔楼': ['塔楼'],
                    '板塔结合': ['板塔', '塔板', '塔板结合'],
                    '平房': ['平房'],
                    '混合类型': ['混合', '多种']
                }
            },
            '房屋用途': {
                'standard_categories': ['住宅', '别墅', '公寓', '商业办公', '商住两用', '车库', '酒店式公寓', 
                                      '花园洋房', '四合院', '写字楼', '底商', '经济适用房', '安置房', '其他'],
                'mapping_rules': {
                    '住宅': ['普通住宅', '住宅', '公寓/住宅', '公寓（住宅）', '住宅式公寓'],
                    '别墅': ['别墅'],
                    '公寓': ['公寓', '公寓/公寓', '老公寓', '商务型公寓', '商务公寓'],
                    '商业办公': ['商业办公类', '商业', '写字楼'],
                    '商住两用': ['商住两用'],
                    '车库': ['车库'],
                    '酒店式公寓': ['酒店式公寓'],
                    '花园洋房': ['花园洋房'],
                    '四合院': ['四合院'],
                    '底商': ['底商'],
                    '经济适用房': ['一类经济适用房', '二类经济适用房', '经济适用房'],
                    '安置房': ['定向安置房', '拆迁还建房', '动迁安置房', '回迁房'],
                    '其他': ['新式里弄', '其他', '未知']
                }
            },
            '交易权属': {
                'standard_categories': ['商品房', '已购公房', '经济适用房', '限价商品房', '央产房', '私产', 
                                      '安置房', '房改房', '集资房', '售后公房', '其他'],
                'mapping_rules': {
                    '商品房': ['商品房'],
                    '已购公房': ['已购公房'],
                    '经济适用房': ['一类经济适用房', '二类经济适用房', '经济适用房'],
                    '限价商品房': ['限价商品房'],
                    '央产房': ['央产房'],
                    '私产': ['私产'],
                    '安置房': ['定向安置房', '拆迁还建房', '动迁安置房', '回迁房'],
                    '房改房': ['房改房'],
                    '集资房': ['集资房'],
                    '售后公房': ['售后公房'],
                    '其他': ['其他', '未知']
                }
            }
        }
    
    def intelligent_category_mapping(self, text, feature_name):
        """智能分类映射"""
        if pd.isna(text):
            return '未知'
        
        text = str(text).lower().strip()
        standards = self.create_standard_categories().get(feature_name, {})
        
        if not standards:
            return text  # 如果没有定义标准，返回原文本
        
        # 精确匹配
        for standard_cat, keywords in standards.get('mapping_rules', {}).items():
            for keyword in keywords:
                if keyword.lower() in text:
                    return standard_cat
        
        # 模糊匹配
        return self.fuzzy_match(text, standards.get('standard_categories', []))
    
    def fuzzy_match(self, text, candidates):
        """模糊匹配"""
        text = str(text).lower()
        for candidate in candidates:
            if candidate.lower() in text or text in candidate.lower():
                return candidate
        return '其他'
    
    def handle_composite_categories(self, series, feature_name):
        """处理复合分类（如'板楼/塔楼'）"""
        def split_composite(text):
            if pd.isna(text):
                return ['未知']
            
            text = str(text)
            # 多种分隔符
            separators = ['/', '、', '及', '和', '与', '\\']
            for sep in separators:
                if sep in text:
                    return [part.strip() for part in text.split(sep) if part.strip()]
            return [text]
        
        # 展开复合类别
        exploded = series.apply(split_composite).explode()
        
        # 标准化每个部分
        standardized_parts = exploded.apply(
            lambda x: self.intelligent_category_mapping(x, feature_name)
        )
        
        # 重新组合（去重并排序）
        def recombine(groups):
            unique_cats = sorted(set(groups))
            return '/'.join(unique_cats) if len(unique_cats) > 1 else unique_cats[0]
        
        result = standardized_parts.groupby(level=0).apply(recombine)
        return result

    def analyze_category_distribution(self, series, feature_name, dataset_name=""):
        """分析分类分布"""
        dataset_prefix = f"{dataset_name} " if dataset_name else ""
        print(f"\n=== {dataset_prefix}{feature_name} 分类分析 ===")
        value_counts = series.value_counts(dropna=False)
        print("原始分布:")
        for value, count in value_counts.items():
            print(f"  {value}: {count} ({count/len(series)*100:.1f}%)")
        
        # 清洗后的分布
        if feature_name in ['建筑结构_comm']:  # 只有建筑结构_comm需要复合处理
            cleaned = self.handle_composite_categories(series, feature_name)
        else:
            cleaned = series.apply(lambda x: self.intelligent_category_mapping(x, feature_name))
        
        cleaned_counts = cleaned.value_counts()
        print("\n清洗后分布:")
        for value, count in cleaned_counts.items():
            print(f"  {value}: {count} ({count/len(series)*100:.1f}%)")
        
        return cleaned

    def fit_label_encoders(self, df_cleaned):
        """训练Label Encoder"""
        self.label_encoders = {}
        for column in df_cleaned.columns:
            le = LabelEncoder()
            le.fit(df_cleaned[column])
            self.label_encoders[column] = le
        
        print("\n=== Label Encoding 映射关系 ===")
        for column, le in self.label_encoders.items():
            print(f"\n{column}:")
            for i, class_name in enumerate(le.classes_):
                print(f"  {class_name} -> {i}")
        
        self.fitted = True
        return self.label_encoders

    def transform_labels(self, df_cleaned, replace_original=True):
        """将分类转换为数字标签，直接替代原始列"""
        if not self.fitted:
            self.fit_label_encoders(df_cleaned)
        
        df_encoded = df_cleaned.copy()
        for column in df_encoded.columns:
            if column in self.label_encoders:
                # 处理未知标签
                known_classes = set(self.label_encoders[column].classes_)
                current_classes = set(df_encoded[column].unique())
                
                unknown_classes = current_classes - known_classes
                if unknown_classes:
                    print(f"警告: {column} 列发现未知类别: {unknown_classes}")
                    # 将未知类别映射为'其他'
                    df_encoded[column] = df_encoded[column].apply(
                        lambda x: x if x in known_classes else '其他'
                    )
                
                # 重新训练编码器以包含'其他'类别
                if unknown_classes:
                    le = LabelEncoder()
                    le.fit(df_encoded[column])
                    self.label_encoders[column] = le
                
                # 直接替换原始列，而不是创建新列
                encoded_values = self.label_encoders[column].transform(df_encoded[column])
                df_encoded[column] = encoded_values
                
                # 如果需要，也可以保留原始文本列（添加后缀）
                if not replace_original:
                    df_encoded[f'{column}_original'] = df_cleaned[column]
        
        return df_encoded

    def save_cleaner(self, filepath):
        """保存清洗器"""
        with open(filepath, 'wb') as f:
            pickle.dump({
                'label_encoders': self.label_encoders,
                'fitted': self.fitted
            }, f)
        print(f"清洗器已保存到: {filepath}")

    def load_cleaner(self, filepath):
        """加载清洗器"""
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
            self.label_encoders = data['label_encoders']
            self.fitted = data['fitted']
        print(f"清洗器已从 {filepath} 加载")

def process_price_datasets(price_train, price_test):
    """处理price_train和price_test数据集的特征列"""
    print("=" * 80)
    print("开始处理房价数据集的特征列")
    print("=" * 80)
    
    # 初始化清洗器
    cleaner = TextCategoryCleaner()
    
    # 定义要清洗的特征列
    features_to_clean = ['环线', '聚类层级', '建筑结构', '供暖', '建筑结构_comm', '房屋用途', '交易权属']
    
    # 只处理数据集中实际存在的特征
    available_features = [f for f in features_to_clean if f in price_train.columns]
    print(f"将清洗的特征: {available_features}")
    
    # 1. 处理训练集
    print("\n" + "=" * 60)
    print("处理训练集 (price_train)")
    print("=" * 60)
    
    # 清洗训练集的特征列
    train_cleaned_data = {}
    for feature in available_features:
        train_cleaned_data[feature] = cleaner.analyze_category_distribution(
            price_train[feature], feature, "训练集"
        )
    
    # 创建清洗后的训练集 - 只替换特征列，其他列保持不变
    price_train_cleaned = price_train.copy()
    for feature in available_features:
        price_train_cleaned[feature] = train_cleaned_data[feature]
    
    print(f"\n训练集原始形状: {price_train.shape}")
    print(f"训练集清洗后形状: {price_train_cleaned.shape}")
    
    # 2. 使用训练集拟合编码器
    print("\n" + "=" * 60)
    print("使用训练集拟合编码器")
    print("=" * 60)
    
    # 只使用清洗后的分类特征来拟合编码器
    train_features_for_encoding = price_train_cleaned[available_features]
    price_train_encoded_features = cleaner.transform_labels(train_features_for_encoding, replace_original=True)
    
    # 将编码后的特征替换回原始训练集
    for feature in available_features:
        price_train_cleaned[feature] = price_train_encoded_features[feature]
    
    # 3. 处理测试集
    print("\n" + "=" * 60)
    print("处理测试集 (price_test)")
    print("=" * 60)
    
    # 清洗测试集的特征列（使用训练集学到的规则）
    test_cleaned_data = {}
    for feature in available_features:
        if feature in price_test.columns:
            test_cleaned_data[feature] = price_test[feature].apply(
                lambda x: cleaner.intelligent_category_mapping(x, feature)
            )
    
    # 创建清洗后的测试集 - 只替换特征列，其他列保持不变
    price_test_cleaned = price_test.copy()
    for feature in available_features:
        if feature in price_test.columns:
            price_test_cleaned[feature] = test_cleaned_data[feature]
    
    print(f"\n测试集原始形状: {price_test.shape}")
    print(f"测试集清洗后形状: {price_test_cleaned.shape}")
    
    # 4. 使用训练集学到的编码器转换测试集
    print("\n" + "=" * 60)
    print("使用训练集编码器转换测试集")
    print("=" * 60)
    
    # 只使用清洗后的分类特征来转换
    test_features_for_encoding = price_test_cleaned[available_features]
    price_test_encoded_features = cleaner.transform_labels(test_features_for_encoding, replace_original=True)
    
    # 将编码后的特征替换回原始测试集
    for feature in available_features:
        price_test_cleaned[feature] = price_test_encoded_features[feature]
    
    # 5. 显示数据质量提升统计
    print("\n" + "=" * 60)
    print("数据质量提升统计")
    print("=" * 60)
    
    for feature in available_features:
        if feature in price_train.columns:
            # 训练集统计
            train_original_unique = price_train[feature].nunique()
            train_cleaned_unique = train_cleaned_data[feature].nunique()
            train_improvement = (1 - train_cleaned_unique/train_original_unique) * 100 if train_original_unique > 0 else 0
            
            print(f"\n{feature} - 训练集:")
            print(f"  原始唯一值数量: {train_original_unique}")
            print(f"  清洗后唯一值数量: {train_cleaned_unique}")
            print(f"  分类简化: {train_improvement:.1f}%")
            print(f"  主要类别: {list(train_cleaned_data[feature].value_counts().head(3).index)}")
        
        if feature in price_test.columns:
            # 测试集统计
            test_original_unique = price_test[feature].nunique()
            test_cleaned_unique = test_cleaned_data[feature].nunique() if feature in test_cleaned_data else 0
            test_improvement = (1 - test_cleaned_unique/test_original_unique) * 100 if test_original_unique > 0 else 0
            
            print(f"\n{feature} - 测试集:")
            print(f"  原始唯一值数量: {test_original_unique}")
            print(f"  清洗后唯一值数量: {test_cleaned_unique}")
            print(f"  分类简化: {test_improvement:.1f}%")
            if feature in test_cleaned_data:
                print(f"  主要类别: {list(test_cleaned_data[feature].value_counts().head(3).index)}")
    
    # 6. 保存清洗器
    print("\n" + "=" * 60)
    print("保存清洗器")
    print("=" * 60)
    
    cleaner.save_cleaner('price_category_cleaner.pkl')
    
    # 7. 最终数据集信息
    print("\n" + "=" * 60)
    print("最终数据集信息")
    print("=" * 60)
    
    print(f"训练集最终形状: {price_train_cleaned.shape}")
    print(f"测试集最终形状: {price_test_cleaned.shape}")
    
    print("\n训练集编码特征示例:")
    for feature in available_features:
        if feature in price_train_cleaned.columns:
            unique_vals = price_train_cleaned[feature].nunique()
            data_type = price_train_cleaned[feature].dtype
            print(f"  {feature}: {unique_vals} 个唯一值, 数据类型: {data_type}")
    
    print("\n测试集编码特征示例:")
    for feature in available_features:
        if feature in price_test_cleaned.columns:
            unique_vals = price_test_cleaned[feature].nunique()
            data_type = price_test_cleaned[feature].dtype
            print(f"  {feature}: {unique_vals} 个唯一值, 数据类型: {data_type}")
    
    # 8. 显示编码映射关系
    print("\n" + "=" * 60)
    print("编码映射关系")
    print("=" * 60)
    
    for feature in available_features:
        if feature in cleaner.label_encoders:
            print(f"\n{feature}:")
            le = cleaner.label_encoders[feature]
            for i, class_name in enumerate(le.classes_):
                print(f"  {class_name} -> {i}")
    
    return price_train_cleaned, price_test_cleaned, cleaner

def apply_cleaner_to_new_data(new_df, cleaner_path='price_category_cleaner.pkl', replace_original=True):
    """对新数据应用已训练的清洗器"""
    cleaner = TextCategoryCleaner()
    cleaner.load_cleaner(cleaner_path)
    
    # 获取清洗器支持的特征列
    available_features = list(cleaner.label_encoders.keys())
    
    # 清洗新数据
    cleaned_data = {}
    for column in available_features:
        if column in new_df.columns:
            if column == '建筑结构_comm':
                cleaned_data[column] = cleaner.handle_composite_categories(new_df[column], column)
            else:
                cleaned_data[column] = new_df[column].apply(
                    lambda x: cleaner.intelligent_category_mapping(x, column)
                )
    
    # 创建清洗后的数据
    df_cleaned = new_df.copy()
    for column in cleaned_data.keys():
        df_cleaned[column] = cleaned_data[column]
    
    # 编码
    features_for_encoding = df_cleaned[list(cleaned_data.keys())]
    df_encoded = cleaner.transform_labels(features_for_encoding, replace_original=replace_original)
    
    # 将编码后的特征替换回去
    for column in cleaned_data.keys():
        df_cleaned[column] = df_encoded[column]
    
    return df_cleaned

# 使用示例
if __name__ == "__main__":
    
    # 处理数据集 - 只处理特征列，不涉及目标变量
    price_train_cleaned, price_test_cleaned, cleaner = process_price_datasets(
        price_train, price_test
    )
    

开始处理房价数据集的特征列
将清洗的特征: ['环线', '聚类层级', '建筑结构', '供暖', '建筑结构_comm', '房屋用途', '交易权属']

处理训练集 (price_train)

=== 训练集 环线 分类分析 ===
原始分布:
  内环内: 47066 (45.3%)
  外环外: 12439 (12.0%)
  六环外: 9391 (9.0%)
  五至六环: 9326 (9.0%)
  内环至外环: 6221 (6.0%)
  内环至中环: 6036 (5.8%)
  中环至外环: 4823 (4.6%)
  三至四环: 2829 (2.7%)
  四至五环: 2677 (2.6%)
  二至三环: 2509 (2.4%)
  二环内: 554 (0.5%)

清洗后分布:
  内环内: 47066 (45.3%)
  外环外: 12439 (12.0%)
  六环外: 9391 (9.0%)
  五至六环: 9326 (9.0%)
  其他: 6221 (6.0%)
  内环至中环: 6036 (5.8%)
  中环至外环: 4823 (4.6%)
  三至四环: 2829 (2.7%)
  四至五环: 2677 (2.6%)
  二至三环: 2509 (2.4%)
  二环内: 554 (0.5%)

=== 训练集 聚类层级 分类分析 ===
原始分布:
  区域级: 103858 (100.0%)
  板块级: 13 (0.0%)

清洗后分布:
  区域级: 103858 (100.0%)
  板块级: 13 (0.0%)

=== 训练集 建筑结构 分类分析 ===
原始分布:
  钢混结构: 82519 (79.4%)
  混合结构: 8971 (8.6%)
  未知结构: 4808 (4.6%)
  砖混结构: 4201 (4.0%)
  框架结构: 1676 (1.6%)
  钢结构: 1066 (1.0%)
  nan: 580 (0.6%)
  砖木结构: 50 (0.0%)

清洗后分布:
  钢混结构: 82519 (79.4%)
  混合结构: 8971 (8.6%)
  未知结构: 4808 (4.6%)
  砖混结构: 4201 (4.0%)
  框架结构: 1676 (1.6%)
  钢结构: 106

In [24]:
price_train=price_train_cleaned
price_test=price_test_cleaned

In [25]:
class DeveloperPropertyProcessor:
    def __init__(self, min_count=30, n_components=5, n_clusters=10, target_col="房屋总数", random_state=111):
        """
        参数:
            min_count: 高频类最小出现次数
            n_components: PCA降维维度
            n_clusters: 聚类数量
            target_col: 用于统计特征的目标列
        """
        self.min_count = min_count
        self.n_components = n_components
        self.n_clusters = n_clusters
        self.target_col = target_col
        self.random_state = random_state
        
        # 存储模型和映射
        self.dev_highfreq = None
        self.prop_highfreq = None
        self.tfidf_dev = None
        self.tfidf_prop = None
        self.pca_dev = None
        self.pca_prop = None
        self.kmeans_dev = None
        self.kmeans_prop = None
        self.dev_stats = None
        self.prop_stats = None

    def clean_name(self, name):
        """文本清洗"""
        unknown_patterns = [
            "无开发商", "无", "暂无", "未知", "未公布", "待定", 
            "自建", "个人", "暂无信息", "未知开发商", "开发商待定",
            "无主", "不详", "不清楚", "未明确", "待确认"
        ]
        if pd.isna(name) or name in unknown_patterns:
            return "未知"
        name = str(name)
        name = re.sub(r"[（）()]", "", name)
        name = re.sub(r"(股份|有限|责任|集团|公司|房地产|开发|物业|管理|服务|控股|建设|实业|投资|发展|地产)", "", name)
        return name.strip().lower()

    def fit(self, df):
        """在训练集上拟合"""
        df["开发商_清洁"] = df["开发商"].apply(self.clean_name)
        df["物业公司_清洁"] = df["物业公司"].apply(self.clean_name)

        #    
        print("开发商清洁后唯一值数量:", df["开发商_清洁"].nunique())
        print("开发商清洁后样本:", df["开发商_清洁"].value_counts().head())

        # 高频类统计
        dev_counts = df["开发商_清洁"].value_counts()
        prop_counts = df["物业公司_清洁"].value_counts()
        self.dev_highfreq = set(dev_counts[dev_counts > self.min_count].index)
        self.prop_highfreq = set(prop_counts[prop_counts > self.min_count].index)

        # 统计特征
        if self.target_col in df.columns:
            self.dev_stats = df.groupby("开发商_清洁")[self.target_col].agg(['mean', 'median', 'std', 'count'])
            self.prop_stats = df.groupby("物业公司_清洁")[self.target_col].agg(['mean', 'median', 'std', 'count'])

        # TF-IDF + PCA
        self.tfidf_dev = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
        self.tfidf_prop = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
        X_dev = self.tfidf_dev.fit_transform(df["开发商_清洁"])
        X_prop = self.tfidf_prop.fit_transform(df["物业公司_清洁"])
        self.pca_dev = PCA(n_components=self.n_components, random_state=self.random_state)
        self.pca_prop = PCA(n_components=self.n_components, random_state=self.random_state)
        X_dev_pca = self.pca_dev.fit_transform(X_dev.toarray())
        X_prop_pca = self.pca_prop.fit_transform(X_prop.toarray())

        # 聚类
        self.kmeans_dev = KMeans(n_clusters=self.n_clusters, random_state=self.random_state)
        self.kmeans_prop = KMeans(n_clusters=self.n_clusters, random_state=self.random_state)
        self.kmeans_dev.fit(X_dev_pca)
        self.kmeans_prop.fit(X_prop_pca)
        return self

    def transform(self, df):
        """在训练集或测试集上应用已拟合的映射"""
        df = df.copy()
        df["开发商_清洁"] = df["开发商"].apply(self.clean_name)
        df["物业公司_清洁"] = df["物业公司"].apply(self.clean_name)

        # TF-IDF + PCA + 聚类
        X_dev = self.tfidf_dev.transform(df["开发商_清洁"])
        X_prop = self.tfidf_prop.transform(df["物业公司_清洁"])
        dev_pca = self.pca_dev.transform(X_dev.toarray())
        prop_pca = self.pca_prop.transform(X_prop.toarray())

        # 聚类编号特征
        dev_clusters = self.kmeans_dev.predict(dev_pca)
        prop_clusters = self.kmeans_prop.predict(prop_pca)
        df["开发商_cluster"] = dev_clusters
        df["物业公司_cluster"] = prop_clusters

        # 用聚类结果替换原始列
        df["开发商"] = dev_clusters
        df["物业公司"] = prop_clusters

        # 删除所有中间列
        columns_to_drop = ['开发商_清洁', '物业公司_清洁','开发商_cluster','物业公司_cluster']
        df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])
        return df


In [26]:
processor = DeveloperPropertyProcessor(
    min_count=30,
    n_components=5,
    n_clusters=10,
    target_col="房屋总数"
)

# === 1️⃣ 训练阶段 ===
processor.fit(price_train)
price_train = processor.transform(price_train)

# === 2️⃣ 测试集映射 ===
price_test = processor.transform(price_test)


开发商清洁后唯一值数量: 1960
开发商清洁后样本: 开发商_清洁
未知        41449
三河市莲荷       927
三河顺通        548
陕西亿润经贸      459
重庆汇东        445
Name: count, dtype: int64


In [27]:
def convert_object(train_df, test_df, columns):
    #剩余的简单的文本提取
    missing_fill_map = {
        '装修情况':'其他',
        '别墅类型': '无',
        '房屋年限': '未知',
        '供水': '未知',
        '供电': '未知',
        '政策周期':'未知',
        '聚类层级':'默认级'
    }
    for col in columns:
        if col not in train_df.columns:
            continue
        fill_value = missing_fill_map.get(col, '未知')
        train_df[col] = train_df[col].fillna(fill_value)
        test_df[col] = test_df[col].fillna(fill_value)
    
        le = LabelEncoder()
        le.fit(train_df[col].fillna("__NA__"))
        mapping = dict(zip(le.classes_, le.transform(le.classes_)))
        train_df[col] = train_df[col].fillna("__NA_").map(mapping)
        test_df[col] = test_df[col].fillna("__NA__").map(lambda x: mapping.get(x, -1))
    
    return train_df, test_df

columns=['别墅类型','房屋年限','供水','供电','政策周期','聚类层级','装修情况']
price_train, price_test=convert_object(price_train, price_test, columns)

In [28]:
knn_cols = ['套内面积','供热费','停车费用','小区平均房龄','物业费','卧室数','厨房数','卫生间数','停车位','绿化率','物业费']
knn_cols = [c for c in knn_cols if c in price_train.columns]
date_cols = ['交易时间','上次交易']
date_cols = [c for c in date_cols if c in price_train.columns]

if knn_cols:
    print(f"🔍 使用 sklearn KNNImputer 插值列: {knn_cols}")
    
    # 使用sklearn的KNNImputer（更节省内存）
    knn_imputer = KNNImputer(n_neighbors=5, weights='uniform')
    train_knn = knn_imputer.fit_transform(price_train[knn_cols])
    test_knn = knn_imputer.transform(price_test[knn_cols])
    
    price_train[knn_cols] = train_knn
    price_test[knn_cols] = test_knn

# 处理日期列
for col in date_cols:
    missing_count = price_train[col].isna().sum()
    print(f"  {col}: 训练集缺失值 {missing_count}/{len(price_train)}")
    missing_count_test = price_test[col].isna().sum()
    print(f"  {col}: 测试集缺失值 {missing_count_test}/{len(price_test)}")

# 其他数值列 → 中位数填充
remaining_num_cols = [col for col in num_cols if col not in knn_cols and col not in date_cols]

for col in remaining_num_cols:
    med = price_train[col].median()  
    price_train[col] = price_train[col].fillna(med)
    price_test[col] = price_test[col].fillna(med)

🔍 使用 sklearn KNNImputer 插值列: ['套内面积', '供热费', '停车费用', '小区平均房龄', '物业费', '卧室数', '厨房数', '卫生间数', '停车位', '绿化率', '物业费']
  交易时间: 训练集缺失值 0/103871
  交易时间: 测试集缺失值 0/34017
  上次交易: 训练集缺失值 25466/103871
  上次交易: 测试集缺失值 7243/34017


In [29]:
price_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103871 entries, 0 to 103870
Data columns (total 58 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   城市         103871 non-null  int64         
 1   区域         103871 non-null  float64       
 2   板块         103871 non-null  float64       
 3   环线         103871 non-null  int32         
 4   房屋户型       103291 non-null  object        
 5   所在楼层       103871 non-null  object        
 6   建筑面积       103871 non-null  float64       
 7   套内面积       103871 non-null  float64       
 8   建筑结构       103871 non-null  int32         
 9   装修情况       103871 non-null  int32         
 10  梯户比例       103871 non-null  float64       
 11  配备电梯       103871 non-null  int32         
 12  别墅类型       103871 non-null  int32         
 13  交易时间       103871 non-null  datetime64[ns]
 14  交易权属       103871 non-null  int32         
 15  上次交易       78405 non-null   datetime64[ns]
 16  房屋用途       103871 no

In [30]:
def house_evaluation(df,bed_weight=0.6,liv_weight=0.1,kit_weight=0.2,bath_weight=0.1):
    df['户型评分']=df['卧室数']*bed_weight+df['客厅数']*liv_weight+df['厨房数']*kit_weight+df['卫生间数']*bath_weight
    return df

price_train= house_evaluation(price_train)
price_test=house_evaluation(price_test)

In [31]:
# 添加Price列回训练集
price_train['Price'] = target_Price.values

# 添加ID列回测试集  
price_test['ID'] = price_test_id.values

# 保存
price_train.to_csv('price_train_processed.csv', index=False)
price_test.to_csv('price_test_processed.csv', index=False)

print("✓ 数据已保存")


✓ 数据已保存
